<a href="https://colab.research.google.com/github/par21val-512/smbpls/blob/main/smbpls_scvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
! pip install --quiet anndata mudata scvi-colab

from scvi_colab import install
install()

INFO     scvi-colab: Installing scvi-tools.                                                                        
INFO     scvi-colab: Install successful. Testing import.                                                           


/usr/local/lib/python3.12/dist-packages/pyro/ops/stats.py:527: SyntaxWarning: invalid escape sequence '\g'
  we have :math:`ES^{*}(P,Q) \ge ES^{*}(Q,Q)` with equality holding if and only if :math:`P=Q`, i.e.


In [5]:
import os
import tempfile

import scanpy as sc
import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [6]:
import torch
from torch import nn
import numpy as np

import sklearn

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn import datasets
import matplotlib.pyplot as plt

import warnings
import anndata as ad
import mudata as mu
import pandas as pd

RANDOM_SEED = 42

def generate_mudata(RANDOM_SEED, stdev_scaler=0.0):
    torch.manual_seed(RANDOM_SEED)

    n = 1000
    p1, p2, p3 = 50, 80, 30
    k = 2

    z = torch.normal(mean=0, std=1, size=(n, k))

    true_w1 = torch.randn(p1, k)
    true_w2 = torch.randn(p2, k)
    true_w3 = torch.randn(p3, k)

    x1 = z @ true_w1.T + stdev_scaler * torch.randn(n, p1)
    x2 = z @ true_w2.T + stdev_scaler * torch.randn(n, p2)
    x3 = z @ true_w3.T + stdev_scaler * torch.randn(n, p3)

    # Sparse latent
    z_sparse = z.clone()
    z_sparse[:500, 0] = 0
    z_sparse[500:, 1] = 0

    x1_sparse = z_sparse @ true_w1.T + stdev_scaler * torch.randn(n, p1)
    x2_sparse = z_sparse @ true_w2.T + stdev_scaler * torch.randn(n, p2)
    x3_sparse = z_sparse @ true_w3.T + stdev_scaler * torch.randn(n, p3)

    # Train/test split
    indices = np.arange(n)
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=RANDOM_SEED
    )

    split_labels = np.array(["train"] * n)
    split_labels[test_idx] = "test"

    # Helper to build AnnData
    def build_adata(X, X_sparse, name):
        adata = ad.AnnData(X=X.numpy())
        adata.layers["sparse"] = X_sparse.numpy()
        adata.obs["split"] = split_labels
        adata.var["feature"] = [f"{name}_{i}" for i in range(X.shape[1])]
        return adata

    adata1 = build_adata(x1, x1_sparse, "mod1")
    adata2 = build_adata(x2, x2_sparse, "mod2")
    adata3 = build_adata(x3, x3_sparse, "mod3")

    mdata = mu.MuData({
        "mod1": adata1,
        "mod2": adata2,
        "mod3": adata3,
    })

    return mdata

In [7]:
mdata = generate_mudata(RANDOM_SEED)

/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:571: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  self._update_attr_legacy(attr, axis, join_common, **kwargs)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility

In [9]:
import torch
import torch.nn as nn
from scvi.module.base import BaseModuleClass
from scvi.distributions import Normal


class SMBPLSModule(BaseModuleClass):
    def __init__(
        self,
        n_input_per_modality,
        n_components=2,
        lam_w=0.05,
        sparsity_freq=10,
    ):
        super().__init__()

        self.block_names = list(n_input_per_modality.keys())
        self.K = n_components
        self.lam_w = lam_w
        self.sparsity_freq = sparsity_freq
        self._step_count = 0

        # Linear projections per modality
        self.proj = nn.ModuleDict({
            mod: nn.Linear(n_input_per_modality[mod], self.K, bias=False)
            for mod in self.block_names
        })

        # Regression head
        self.regressor = nn.Linear(self.K, 1)

        # Learned observation noise
        self.log_sigma = nn.Parameter(torch.zeros(1))

    # ---------- Proximal L1 step ----------
    @torch.no_grad()
    def apply_weight_sparsity(self):
        for mod in self.block_names:
            W = self.proj[mod].weight
            W.copy_(torch.sign(W) * torch.clamp(torch.abs(W) - self.lam_w, min=0.0))

    # ---------- Forward ----------
    def forward(self, tensors):
        t = None

        for mod in self.block_names:
            X = tensors[mod]
            tb = self.proj[mod](X)
            t = tb if t is None else t + tb

        y = tensors["y"]

        y_hat = self.regressor(t).squeeze(-1)
        sigma = torch.exp(self.log_sigma)

        dist = Normal(y_hat, sigma)
        log_prob = dist.log_prob(y).sum()

        loss = -log_prob

        # Proximal sparsity step
        self._step_count += 1
        if self.lam_w > 0 and self._step_count % self.sparsity_freq == 0:
            self.apply_weight_sparsity()

        return {
            "loss": loss,
            "y_hat": y_hat,
            "latent": t,
        }